In [1]:
# =========================
# 0. Install dependencies
# =========================

!pip install -q transformers scikit-learn pandas tqdm torch


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =========================
# 1. Imports
# =========================

import os
import re
import json
import sys
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

In [3]:
# =========================
# 2. Config
# =========================

os.chdir(r"C:\Users\aiger\Documents\2026SS\AIR\git_personal\AIR_Group_Task")

RAW_TRAIN_PATH = "data/english/english_train.json"
VAL_PATH = "data/english/clef2026_gpt4_o_mini_val.json"

TRAIN_JSONL = "output/training_data_for_RM/english_train.jsonl"
MODEL_DIR = "output/small_bert_l2"
PRED_PATH = "output/RM_prediction/bert_l2_predictions.json"
RESULT_DIR = "output/results_bert_l2"
BASE_MODEL = "google/bert_uncased_L-2_H-128_A-2"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
RANDOM_STATE = 42

os.makedirs("output/training_data_for_RM", exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("output/RM_prediction", exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [4]:
# =========================
# 3. Reuse provided preprocessing
# =========================
'''
subprocess.run(
    [
        sys.executable,
        "task2/reasoning_trace_build.py",
        "--input",
        RAW_TRAIN_PATH,
        "--output",
        TRAIN_JSONL,
    ],
    check=True,
)
'''
train_df = pd.read_json(TRAIN_JSONL, lines=True)

print(train_df.head())
print(train_df["Class"].value_counts())

  sample_id                                         input_text        Label  \
0       0_a  Claim: “The first randomized controlled trial ...  conflicting   
1       0_b  Claim: “The first randomized controlled trial ...  conflicting   
2       0_c  Claim: “The first randomized controlled trial ...  conflicting   
3       0_d  Claim: “The first randomized controlled trial ...  conflicting   
4       0_e  Claim: “The first randomized controlled trial ...  conflicting   

  Verdict  Class  
0   false      0  
1   false      0  
2   false      0  
3   false      0  
4   false      0  
Class
0    22775
1     8658
Name: count, dtype: int64


In [5]:
# =========================
# 4. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item

In [6]:
# =========================
# 5. Small verifier model
# =========================

class SmallVerifier(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        hidden_size = self.model.config.hidden_size
        self.classifier = torch.nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled_output.float())
        return logits

In [7]:
def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print(f"trainable params: {trainable}")
    print(f"total params: {total}")
    print(f"trainable%: {100 * trainable / total:.2f}")

In [8]:
# =========================
# 6. Trainer
# =========================

class TrainerModule:
    def __init__(self, model, train_loader, val_loader, epochs, lr, output_dir):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(labels.detach().cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()

                preds = (torch.sigmoid(logits).squeeze(1) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(labels.detach().cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model_epoch_{epoch}.pt"),
        )

In [9]:
#import sys
#!{sys.executable} -m pip install sentencepiece

In [10]:
# =========================
# 7. Train small verifier
# =========================

train_split, dev_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=RANDOM_STATE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
dev_dataset = TextDataset(dev_split, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)

model = SmallVerifier(BASE_MODEL)
print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=dev_loader,
    epochs=EPOCHS,
    lr=LR,
    output_dir=MODEL_DIR,
)

trainer.train()

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 4386049
total params: 4386049
trainable%: 100.00

Epoch 1/3


100%|██████████| 1572/1572 [09:41<00:00,  2.71it/s]


Train Loss: 0.5915
Train Acc: 0.7099
Val Loss: 0.5410
Val Acc: 0.7453

Epoch 2/3


100%|██████████| 1572/1572 [09:43<00:00,  2.69it/s]


Train Loss: 0.5002
Train Acc: 0.7810
Val Loss: 0.4801
Val Acc: 0.7992

Epoch 3/3


100%|██████████| 1572/1572 [10:17<00:00,  2.55it/s]


Train Loss: 0.4773
Train Acc: 0.7995
Val Loss: 0.4741
Val Acc: 0.8042


In [11]:
# =========================
# 8. Prediction helper
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")

In [12]:
class VerifierEvaluator:
    def __init__(self, model_path, tokenizer_path, base_model, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        self.model = SmallVerifier(base_model)
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, verdict, justification, max_length=150):
        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, verdict, justification):
        input_ids, attention_mask = self.encode_input(claim, verdict, justification)

        with torch.no_grad():
            return float(self.model(input_ids, attention_mask).item())

In [13]:
# =========================
# 9. Generate predictions
# =========================

BEST_EPOCH = EPOCHS - 1
MODEL_PATH = os.path.join(MODEL_DIR, f"model_epoch_{BEST_EPOCH}.pt")

with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

evaluator = VerifierEvaluator(
    model_path=MODEL_PATH,
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
)

predictions = []

for idx, sample in enumerate(tqdm(val_data)):
    verdict_list = []
    verifier_score_list = []
    justification_list = []

    for trace_idx in range(len(sample["Reasoning_traces"])):
        justification = remove_label_pattern(
            sample["Reasoning_traces"][trace_idx]
        ).split("Label:")[0]

        verdict = sample["Verdict_list"][trace_idx].lower()

        score = evaluator.score(
            claim=sample["claim"],
            verdict=verdict,
            justification=justification,
        )

        verdict_list.append(sample["Verdict_list"][trace_idx])
        justification_list.append(justification)
        verifier_score_list.append(score)

    best_idx = int(np.argmax(np.array(verifier_score_list)))
    best_verdict = verdict_list[best_idx]

    predictions.append(
        {
            "query_id": sample.get("query_id", idx),
            "Claim": sample["claim"],
            "Label": sample["label"],
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": verdict_list,
            "Reasoning_traces": justification_list,
            "score_list": verifier_score_list,
        }
    )

with open(PRED_PATH, "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4, ensure_ascii=False)

print(f"Saved predictions to {PRED_PATH}")

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1600/1600 [03:28<00:00,  7.66it/s]


Saved predictions to output/RM_prediction/bert_l2_predictions.json


In [14]:
import shutil

shutil.copy(
    PRED_PATH,
    "output/RM_prediction/clef_predictions.json"
)

'output/RM_prediction/clef_predictions.json'

In [15]:
# =========================
# 10. Run provided scorer
# =========================

subprocess.run(
    [sys.executable, "task2/scorer.py"],
    check=True,
)
os.makedirs("output/results_bert_l2", exist_ok=True)

shutil.copy(
    "output/RM_prediction/result.csv",
    f"{RESULT_DIR}/result.csv"
)

shutil.copy(
    "output/RM_prediction/per_sample_ir.csv",
    f"{RESULT_DIR}/per_sample_ir.csv"
)

'output/results_bert_l2/per_sample_ir.csv'